# Absteckung Einfamilienhaus – Absteckdaten, Kontrolle und Absteckprotokoll
**Setting out a detached house: setting-out data (polar method), control measurement, DXF plan and setting-out protocol**

Fictitious example in ETRS89 / UTM zone 32N (EPSG:25832). All coordinates are assumptions.
Steps: 1 control points and design · 2 setting-out data (bearing, direction, distance) · 3 control measurement with tolerance check · 4 sides and diagonals · 5 DXF plan, Excel, charts and protocol.

In [ ]:
import os, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ezdxf
os.makedirs('results', exist_ok=True); os.makedirs('images', exist_ok=True)
GON = 200 / math.pi
def bearing(e1, n1, e2, n2):          # Richtungswinkel t in gon (0..400), clockwise from grid north
    return (math.atan2(e2 - e1, n2 - n1) * GON) % 400
def dist(e1, n1, e2, n2): return math.hypot(e2 - e1, n2 - n1)

## 1 Control points (Festpunkte) and design (Gebäudeplanung)

In [ ]:
fp = pd.DataFrame([('AP1',366200.000,5702800.000,95.214),('AP2',366262.418,5702843.771,95.488),('AP3',366187.906,5702866.052,95.902)],
                  columns=['Pkt','E','N','H'])
# building 12.00 m x 9.50 m, long side parallel to the street (bearing 62.5000 gon)
E0, N0, T, L, Bw = 366221.350, 5702818.420, 62.5, 12.00, 9.50
ux, uy = math.sin(T/GON), math.cos(T/GON)          # unit vector long side
vx, vy = math.sin((T+100)/GON), math.cos((T+100)/GON)  # unit vector short side (right angle)
corners = [(0,0),(L,0),(L,Bw),(0,Bw)]
soll = pd.DataFrame([(f'G{i+1}', E0 + a*ux + b*vx, N0 + a*uy + b*vy, 95.850) for i,(a,b) in enumerate(corners)], columns=['Pkt','E','N','H'])
soll[['E','N']] = soll[['E','N']].round(3)
soll['Art'] = 'Gebäudeecke (OK Bodenplatte 95,850 m)'
fp, soll

## 2 Setting-out data – polar method from AP1, orientation to AP2

In [ ]:
st, ori, chk = fp.iloc[0], fp.iloc[1], fp.iloc[2]
t0 = bearing(st.E, st.N, ori.E, ori.N)
rows = []
for _, p in soll.iterrows():
    t = bearing(st.E, st.N, p.E, p.N); s = dist(st.E, st.N, p.E, p.N)
    rows.append((p.Pkt, p.E, p.N, p.H, round(t,4), round((t - t0) % 400, 4), round(s,3), round(p.H - st.H, 3)))
abst = pd.DataFrame(rows, columns=['Pkt','E','N','H_soll','t_gon','r_gon','s_m','dH_m'])
# orientation check: direction and distance to AP3
t3, s3 = bearing(st.E, st.N, chk.E, chk.N), dist(st.E, st.N, chk.E, chk.N)
print(f'Standpunkt AP1 · Anschluss AP2: t0 = {t0:.4f} gon · Kontrolle AP3: r = {(t3-t0)%400:.4f} gon, s = {s3:.3f} m')
abst

## 3 Control measurement (Kontrollmessung) – tolerance ±10 mm

In [ ]:
rng = np.random.default_rng(45)
dev = rng.normal(0, 0.004, size=(4, 2)); dev[2] = [0.009, -0.008]   # point G3 deliberately outside tolerance
ist = soll[['Pkt','E','N']].copy(); ist['E_ist'] = (ist.E + dev[:,0]).round(3); ist['N_ist'] = (ist.N + dev[:,1]).round(3)
ist['H_ist'] = (soll.H + rng.normal(0, 0.003, 4)).round(3)
ist['dE_mm'] = ((ist.E_ist - ist.E) * 1000).round(0); ist['dN_mm'] = ((ist.N_ist - ist.N) * 1000).round(0)
ist['dL_mm'] = np.hypot(ist.dE_mm, ist.dN_mm).round(0); ist['dH_mm'] = ((ist.H_ist - soll.H) * 1000).round(0)
TOL = 10
ist['Status'] = np.where(ist.dL_mm <= TOL, 'i. O.', 'nachstecken')
ist

## 4 Sides and diagonals (Spannmaße)

In [ ]:
def pts(df, e, n): return {r.Pkt: (r[e], r[n]) for _, r in df.iterrows()}
S, I = pts(soll,'E','N'), pts(ist,'E_ist','N_ist')
pairs = [('G1','G2','Seite'),('G2','G3','Seite'),('G3','G4','Seite'),('G4','G1','Seite'),('G1','G3','Diagonale'),('G2','G4','Diagonale')]
sp = pd.DataFrame([(f'{a}–{b}', k, round(dist(*S[a],*S[b]),3), round(dist(*I[a],*I[b]),3)) for a,b,k in pairs], columns=['Strecke','Art','Soll_m','Ist_m'])
sp['Diff_mm'] = ((sp.Ist_m - sp.Soll_m) * 1000).round(0)
sp['Status'] = np.where(sp.Diff_mm.abs() <= TOL, 'i. O.', 'prüfen')
kz = pd.Series({
 'Absteckpunkte': len(soll), 'Toleranz Lage (mm)': TOL,
 'Punkte i. O.': int((ist.Status=='i. O.').sum()), 'Punkte nachstecken': int((ist.Status!='i. O.').sum()),
 'max. Lageabweichung (mm)': int(ist.dL_mm.max()), 'mittlere Lageabweichung (mm)': round(float(ist.dL_mm.mean()),1),
 'max. Höhenabweichung (mm)': int(ist.dH_mm.abs().max()), 'Spannmaße außerhalb Toleranz': int((sp.Status!='i. O.').sum()),
}, name='Wert', dtype=object); kz.index.name = 'Kennzahl'
print(kz.to_string()); sp

## 5 Results: Excel, CSV, DXF plan, charts

In [ ]:
with pd.ExcelWriter('results/absteckung_einfamilienhaus.xlsx') as xw:
    kz.to_frame().to_excel(xw, sheet_name='Kennzahlen')
    fp.to_excel(xw, sheet_name='Festpunkte', index=False)
    abst.to_excel(xw, sheet_name='Absteckdaten', index=False)
    ist.to_excel(xw, sheet_name='Kontrolle', index=False)
    sp.to_excel(xw, sheet_name='Spannmasse', index=False)
abst.to_csv('results/absteckdaten.csv', index=False, sep=';', decimal=',')
ist.to_csv('results/kontrolle.csv', index=False, sep=';', decimal=',')
sp.to_csv('results/spannmasse.csv', index=False, sep=';', decimal=',')
kz.to_csv('results/kennzahlen.csv', sep=';', decimal=',')
# point file for the total station (Pkt E N H)
pd.concat([fp[['Pkt','E','N','H']], soll[['Pkt','E','N','H']]]).to_csv('results/punktdatei_tachymeter.txt', index=False, sep=' ', header=False, float_format='%.3f')

doc = ezdxf.new('R2010', setup=True); msp = doc.modelspace()
for name, col in (('FESTPUNKTE',1),('GEBAEUDE',5),('ABSTECKPUNKTE',3),('ABSTECKSTRAHLEN',8),('BEMASSUNG',2),('TEXT',7)):
    doc.layers.add(name, color=col)
for _, p in fp.iterrows():
    msp.add_circle((p.E, p.N), 0.4, dxfattribs={'layer':'FESTPUNKTE'})
    msp.add_text(p.Pkt, height=0.6, dxfattribs={'layer':'TEXT'}).set_placement((p.E+0.6, p.N+0.6))
msp.add_lwpolyline([(r.E, r.N) for _, r in soll.iterrows()], close=True, dxfattribs={'layer':'GEBAEUDE'})
for _, p in soll.iterrows():
    msp.add_point((p.E, p.N, p.H), dxfattribs={'layer':'ABSTECKPUNKTE'})
    msp.add_line((st.E, st.N), (p.E, p.N), dxfattribs={'layer':'ABSTECKSTRAHLEN'})
    msp.add_text(p.Pkt, height=0.5, dxfattribs={'layer':'TEXT'}).set_placement((p.E+0.4, p.N+0.4))
for a, b, k in pairs[:2]:
    (e1,n1),(e2,n2) = S[a], S[b]
    msp.add_aligned_dim(p1=(e1,n1), p2=(e2,n2), distance=-1.5, dxfattribs={'layer':'BEMASSUNG'}).render()
msp.add_line((st.E, st.N), (ori.E, ori.N), dxfattribs={'layer':'ABSTECKSTRAHLEN', 'linetype':'DASHED'})
doc.saveas('results/absteckplan_einfamilienhaus.dxf')
print('saved')

In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
E_, N_ = 366150, 5702790   # local origin for readable axes
fig, ax = plt.subplots(figsize=(9, 7.5))
poly = soll[['E','N']].values - [E_, N_]
ax.fill(*np.vstack([poly, poly[:1]]).T, color='#DCE3EE', ec='#1F3864', lw=2, label='Gebäude 12,00 × 9,50 m')
for _, p in soll.iterrows():
    ax.plot([st.E-E_, p.E-E_], [st.N-N_, p.N-N_], color='#999999', lw=0.8, ls='--')
    ax.plot(p.E-E_, p.N-N_, 's', color='#1F3864')
    ax.annotate(p.Pkt, (p.E-E_, p.N-N_), xytext={'G1':(-22,6),'G2':(4,8),'G3':(8,2),'G4':(-24,-12)}[p.Pkt], textcoords='offset points', fontweight='bold')
ax.plot([st.E-E_, ori.E-E_], [st.N-N_, ori.N-N_], color='#C0392B', lw=1.2, label=f'Anschluss AP1 → AP2 (t₀ = {t0:.4f} gon)'.replace('.', ','))
for _, p in fp.iterrows():
    ax.plot(p.E-E_, p.N-N_, '^', color='#C0392B', ms=10); ax.annotate(p.Pkt, (p.E-E_, p.N-N_), xytext=(6, -12), textcoords='offset points', color='#C0392B', fontweight='bold')
for _, r in abst.iterrows():
    p = soll[soll.Pkt == r.Pkt].iloc[0]
    off = {'G1':(-78,8),'G2':(-80,18),'G3':(12,-22),'G4':(8,-28)}[r.Pkt]
    ax.annotate(f'r = {r.r_gon:.4f} gon\ns = {r.s_m:.3f} m'.replace('.', ','), (p.E-E_, p.N-N_), xytext=off, textcoords='offset points', fontsize=7.5, color='#444444')
ax.set_aspect('equal'); ax.set_xlabel(f'E − {E_:,} (m)'.replace(',', '.')); ax.set_ylabel(f'N − {N_:,} (m)'.replace(',', '.'))
ax.legend(frameon=False, loc='lower right', fontsize=8.5); ax.grid(alpha=0.25)
ax.set_title('Absteckplan – polare Absteckung von AP1 (ETRS89 / UTM32)', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/01_Absteckplan.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(6.5, 6))
th = np.linspace(0, 2*np.pi, 200); ax.plot(TOL*np.cos(th), TOL*np.sin(th), color='#555555', ls='--', lw=1, label=f'Toleranz ±{TOL} mm')
for _, r in ist.iterrows():
    c = '#2E8B57' if r.Status == 'i. O.' else '#C0392B'
    ax.plot(r.dE_mm, r.dN_mm, 'o', color=c, ms=9); ax.annotate(f'{r.Pkt} ({r.dL_mm:.0f} mm)', (r.dE_mm, r.dN_mm), xytext={'G2':(-70,8),'G4':(8,6)}.get(r.Pkt,(7,4)), textcoords='offset points', fontsize=9)
ax.axhline(0, color='black', lw=0.6); ax.axvline(0, color='black', lw=0.6)
ax.set_xlim(-15, 15); ax.set_ylim(-15, 15); ax.set_aspect('equal'); ax.set_xlabel('ΔE (mm)'); ax.set_ylabel('ΔN (mm)')
ax.legend(frameon=False, loc='upper left'); ax.set_title('Kontrollmessung: Lageabweichung Ist − Soll', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/02_Kontrolle_Abweichungen.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(8, 4))
cols = ['#2E8B57' if s == 'i. O.' else '#C0392B' for s in sp.Status]
ax.bar(sp.Strecke + '\n' + sp.Art, sp.Diff_mm, color=cols)
for x in (-TOL, TOL): ax.axhline(x, ls='--', color='#555555', lw=1)
ax.axhline(0, color='black', lw=0.6)
for i, v in enumerate(sp.Diff_mm): ax.text(i, v + (0.6 if v >= 0 else -1.4), f'{v:+.0f}', ha='center', fontsize=9)
ax.set_ylim(-TOL-8, TOL+8); ax.set_ylabel('Ist − Soll (mm)'); ax.tick_params(axis='x', labelsize=8.5)
ax.set_title('Spannmaße: Seiten und Diagonalen', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/03_Spannmasse.png', dpi=150); plt.close()
print('charts saved')